# Setup and Imports
Import essential libraries and set up environment configurations.

In [ ]:
import os
# Set HF_HOME externally if you want a custom HF cache location
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import sys
sys.path.insert(0, '..')

from trl import SFTConfig, SFTTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList, set_seed
from trl.trainer.sft_trainer import DataCollatorForLanguageModeling
import torch
import json
from datasets import load_dataset
from accelerate import PartialState

from pipeline.prompts import SYSTEM_PROMPT
from pipeline.training import expand_tokenizer

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
set_seed(2025)

# Model and Tokenizer Initialization
Load the base model and tokenizer, and expand the tokenizer if taking Qwen3-0.6B.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    dtype=torch.bfloat16,
    use_cache = False, # !GradCheckpointing
    device_map={'':PartialState().process_index},
    # quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

if "Qwen3-0.6B" in MODEL_NAME:
    tokenizer, model = expand_tokenizer(tokenizer=tokenizer, model=model)

# Dataset Preparation
Load and format the training data using the prompt template.

In [ ]:
def create_instruction(sample):
    # Build examples string if examples exist
    item = {}
    examples_str = ""
    if sample['examples'] != "NONE":
        examples = json.loads(sample['examples'])
        examples_str_list = [
            f"""<example_note>
{example['original']}
</example_note>

<example_phi_entities>
{example['anonymized']}
</example_phi_entities>""" 
            for example in examples
        ]
        examples_str = "\n\n".join(examples_str_list)
        examples_str = f"<examples>\n{examples_str}\n</examples>\n\n"
    
    return {
        "prompt": \
f"""<instructions>
{SYSTEM_PROMPT}
</instructions>

{examples_str}<note>
{sample['original']}
</note>\n\n""", # \n\n is added to ensure that the last token of prompt is not changed when completion is added
        "completion": sample['anonymized']
    }
    
dataset = load_dataset("csv", data_files={"train": "../data/train.csv", "test": "../data/test.csv"})
cols_to_remove = dataset['train'].features

dataset = dataset.map(
    create_instruction,
    num_proc=4,
    remove_columns=cols_to_remove,
    desc="Applying insruction template",
)

# Training Configuration
Define hyperparameters, calculate stepping for the custom LR scheduler, and initialize the SFTTrainer.

In [ ]:
gradient_accumulation_steps = 8
num_train_epochs = 1
per_device_batch_size = 2
num_devices = torch.cuda.device_count() if "CUDA_VISIBLE_DEVICES" not in os.environ else len(os.environ["CUDA_VISIBLE_DEVICES"].split(","))
total_steps = (len(dataset["train"]) // (per_device_batch_size * num_devices * gradient_accumulation_steps)) * num_train_epochs

warmup_steps = max(100, int(total_steps * 0.05))              # 5% of total steps (at most 100 steps)
stable_steps = int(total_steps * 0.9) - warmup_steps   # 90% of total steps - warmup steps
decay_steps = total_steps - warmup_steps - stable_steps # remaining steps

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(
        pad_token_id=tokenizer.convert_tokens_to_ids(tokenizer.pad_token),
        completion_only_loss=True,
    ),
    args=SFTConfig(
        bf16=True,
        seed=2025,
        data_seed=2025,
        use_liger_kernel=True,
        gradient_accumulation_steps=gradient_accumulation_steps,
        gradient_checkpointing= True,
        gradient_checkpointing_kwargs={'use_reentrant':False},
        optim="adamw_torch_fused",
        learning_rate=5e-5,
        lr_scheduler_type="warmup_stable_decay",
        lr_scheduler_kwargs={'num_stable_steps':stable_steps, 'num_decay_steps':decay_steps},
        warmup_steps=warmup_steps,
        max_steps=total_steps,
        num_train_epochs=num_train_epochs,
        output_dir= f"data/{MODEL_NAME.split('/')[-1]}_SFT",
        run_name = f"{MODEL_NAME.split('/')[-1]}_SFT",
        per_device_eval_batch_size= per_device_batch_size,
        per_device_train_batch_size= per_device_batch_size,
        push_to_hub= False,
        packing=False,
        eval_packing=False,
        max_length=8192,
        do_eval=True,
        logging_strategy="steps",
        logging_steps=4,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_dir = "./log",
        report_to="wandb",
        load_best_model_at_end=False,
    ),
)

# Model Training
Execute the training loop and save the fine-tuned model.

In [ ]:
trainer.train()

# kill all processes except rank 0
if PartialState().is_main_process:
    trainer.save_model(MODEL_NAME.split("/")[-1]+"_SFT")
    print("Model saved in rank 0 at", MODEL_NAME.split("/")[-1]+"_SFT")

# Inference & Evaluation Setup
Define customized stopping criteria to cleanly handle generation halting at expected tags.

In [ ]:
class StoppingCriteriaSub(StoppingCriteria):
    def __init__(self, stops=[], encounters=1):
        super().__init__()
        self.stops = stops
        self.encounters = encounters
        self.matched_counts = {tuple(stop.tolist()): 0 for stop in stops}
    
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
        # Convert input_ids to CPU for easier handling
        input_ids_list = input_ids[0].tolist()  # Assuming batch size of 1
        
        # Check each stop sequence
        for stop in self.stops:
            stop_list = stop.tolist()
            stop_len = len(stop_list)
            
            # If input is long enough to contain the stop sequence
            if len(input_ids_list) >= stop_len:
                # Check if the last tokens match the stop sequence
                if input_ids_list[-stop_len:] == stop_list:
                    self.matched_counts[tuple(stop_list)] += 1
                    if self.matched_counts[tuple(stop_list)] >= self.encounters:
                        return True
        
        return False

# Setup stopping criteria
stop_words = ["\n</phi_entities>", "</phi_entities>"]  # Added both variations
stop_words_ids = []

for stop_word in stop_words:
    # Tokenize without adding special tokens to get exact token ids
    tokens = tokenizer(stop_word, return_tensors='pt', add_special_tokens=False)['input_ids'].squeeze()
    stop_words_ids.append(tokens.to("cuda"))

stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(stops=stop_words_ids)])

# Test Fine-Tuned Model
Load the newly fine-tuned model and test it on a sample clinical note.

In [ ]:
input_text = \
f"""<instructions>
{SYSTEM_PROMPT}
</instructions>

<note>
Patient Name: Johnson, Emily R.
DOB: 09/15/1988
MRN: 7654321
Admission Date: 03/22/2023
Discharge Date: 03/25/2023

Attending Physician: Dr. Amelia Chang

Chief Complaint: Severe lower back pain

History of Present Illness:
Emily Johnson, a 34-year-old female, presented to Sunrise General Hospital on 03/22/2023 with complaints of severe lower back pain that began 3 days prior. Patient reports no recent trauma or injury. Pain is described as constant, sharp, and radiating down the left leg. Patient rates pain as 8/10. Tried over-the-counter pain medications with minimal relief.

Past Medical History:
- Hypothyroidism (diagnosed 2015)
- Seasonal allergies

Medications:
- Levothyroxine 50mcg daily
- Loratadine 10mg as needed

Allergies: Penicillin (hives)

Social History:
Patient works as a software engineer. Lives with husband and two children. Denies tobacco use. Occasional alcohol use (1-2 drinks/week). Exercises 2-3 times per week.

Family History:
Father: Hypertension
Mother: Type 2 Diabetes
No family history of back problems or autoimmune disorders

Physical Examination:
Vitals: BP 128/78, HR 76, RR 16, Temp 98.6°F, SpO2 99% on room air
General: Alert and oriented, in obvious discomfort
Back: Tenderness to palpation in lower lumbar region, limited range of motion due to pain
Neurological: Straight leg raise positive on left at 30 degrees, negative on right. DTRs 2+ bilaterally. Strength 5/5 in all extremities except left ankle dorsiflexion 4/5.

Diagnostic Studies:
- X-ray Lumbar Spine (03/22/2023): No fractures or dislocations. Mild degenerative changes at L4-L5.
- MRI Lumbar Spine (03/23/2023): L4-L5 disc herniation with left-sided nerve root impingement.

Hospital Course:
Emily, was admitted for pain management and further evaluation. Started on IV pain medications and muscle relaxants. Physical therapy and occupational therapy were consulted. Patient showed gradual improvement with conservative management.

Discharge Plan:
1. Medications:
   - Cyclobenzaprine 10mg TID PRN for muscle spasms
   - Naproxen 500mg BID with food for pain
   - Continue home medications

2. Follow-up:
   - Appointment with Dr. Robert Green (Neurosurgery) on 04/05/2023 at 2:00 PM
     Address: 456 Elm Street, Suite 300, Anytown, USA 12345
     Phone: (555) 987-6543

3. Instructions:
   - Continue physical therapy exercises as instructed
   - Use ice/heat therapy as needed
   - Avoid heavy lifting and prolonged sitting
   - Call if symptoms worsen or new symptoms develop

4. Work:
   - Off work until 04/03/2023
   - May return to light duty with restrictions: no lifting >10 lbs, frequent position changes

Patient educated on diagnosis, treatment plan, and follow-up. All questions answered. Patient verbalizes understanding and agrees with the plan.

Discharge Disposition: Home

Electronically signed by:
Amelia Chang, MD
NPI: 1234567890
Date: 03/25/2023 14:30

CC: Dr. Lisa Thompson (Primary Care Physician)
    Email: l.thompson@cityclinic.org
</note>

<phi_entities>
"""

if PartialState().is_main_process:
   model = AutoModelForCausalLM.from_pretrained(
      MODEL_NAME.split("/")[-1]+"_SFT/",
      device_map={'':PartialState().process_index},
      attn_implementation="flash_attention_2",
      dtype=torch.bfloat16,
      use_cache = True, # !GradCheckpointing
      # quantization_config=bnb_config
   )
   tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME.split("/")[-1]+"_SFT/")
   
   with torch.no_grad():
      input_ids = tokenizer([input_text], return_tensors="pt").to("cuda")
      outputs = model.generate(**input_ids, max_new_tokens=2048, do_sample=False, stopping_criteria=stopping_criteria, top_p=1.0, temperature=None)
   print("<phi_entities>")
   print(tokenizer.decode(outputs[0][input_ids['input_ids'].shape[-1]:]))